# Семинар 2 · Разложение и прогнозирование временных рядов

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1pYGd73vX2s3FTsintlwfCqBXMFDWGvdE)

1. STL: компоненты ряда
2. Box–Cox: преобразование шкалы
3. SES, Хольт, Хольт–Уинтерс и обозначения ETS
4. Стационарность, разности, ADF / KPSS
5. ACF / PACF

[Приложение ★](#appendix): дополнительные примеры и задания, если останется время

In [ ]:
%pip install numpy pandas scipy "statsmodels>=0.14.4,<0.15" plotly

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

np.random.seed(42)

## Данные

Месячные продажи магазинов одежды и аксессуаров в США, млн USD, без сезонной корректировки. Первое число обозначает весь месяц

Источник — U.S. Census Bureau через [FRED](https://fred.stlouisfed.org/series/MRTSSM448USN). Берем январь 2012 — август 2026

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MRTSSM448USN"
raw = pd.read_csv(url, parse_dates=["observation_date"])

raw = raw.set_index("observation_date")
sales = raw["MRTSSM448USN"].sort_index()

sales = sales.loc["2012-01":"2026-08"]
sales = sales.asfreq("MS")
sales.name = "sales"
sales.index.name = "Дата"

In [ ]:
period = 12  # Годовой сезон в месячных данных
horizon = 36  # Прогноз на три года

train = sales.iloc[:-horizon]
test = sales.iloc[-horizon:]

split_summary = pd.DataFrame(
    {
        "Начало": [train.index.min(), test.index.min()],
        "Конец": [train.index.max(), test.index.max()],
        "Месяцев": [len(train), len(test)],
    },
    index=["train", "test"],
)
display(split_summary)

In [ ]:
from plotly.subplots import make_subplots


def show_series(frame, title, units="Млн USD", panels=False, split=None):
    """Показывает интерактивный график временных рядов.

    frame — DataFrame с датами в индексе; каждый столбец — отдельный ряд.
    title — заголовок графика.
    units — подпись оси значений, по умолчанию "Млн USD".
    panels — True: ряды на отдельных панелях; False: на одном графике.
    split — дата границы train/test (pd.Timestamp); None: без границы.
    """
    rows = len(frame.columns) if panels else 1
    panel_titles = list(frame.columns) if panels else None

    fig = make_subplots(
        rows=rows,
        cols=1,
        shared_xaxes=True,
        subplot_titles=panel_titles,
        vertical_spacing=0.06 if panels else 0,
    )

    for position, name in enumerate(frame.columns, start=1):
        row = position if panels else 1
        is_actual = name == "Факт"

        fig.add_scatter(
            x=frame.index,
            y=frame[name],
            name=str(name),
            mode="lines",
            line_color="gray" if is_actual else None,
            opacity=0.5 if is_actual else 1,
            row=row,
            col=1,
        )

    if split is not None:
        fig.add_vline(x=split.isoformat(), line_dash="dash")

    fig.update_yaxes(title_text=units)
    fig.update_xaxes(title_text="Дата", row=rows, col=1)
    fig.update_layout(
        title=title,
        height=max(420, 210 * rows),
        showlegend=not panels,
        legend=dict(orientation="h", y=-0.18),
        margin=dict(t=90, b=90, l=80, r=55),
        hovermode="x unified",
    )
    fig.show()

**Функция отрисовки ACF / PACF**

Используем готовую функцию; расчет ACF/PACF будет отдельно в соответствующем разделе

In [ ]:
def show_acf_pacf(acf_result, pacf_result, title):
    """Рисует ACF/PACF по готовым результатам расчета.

    acf_result, pacf_result — оценки и доверительные границы из statsmodels.
    title — заголовок графика.
    """
    results = [acf_result, pacf_result]
    fig = make_subplots(rows=1, cols=2, subplot_titles=["ACF", "PACF"])
    for column, (estimates, interval) in enumerate(results, start=1):
        lags = np.arange(1, len(estimates))
        lower = interval[1:, 0] - estimates[1:]
        upper = interval[1:, 1] - estimates[1:]

        fig.add_scatter(
            x=lags, y=upper, line_width=0, showlegend=False,
            hoverinfo="skip", row=1, col=column,
        )
        fig.add_scatter(
            x=lags, y=lower, line_width=0, fill="tonexty",
            fillcolor="rgba(31,119,180,0.18)", showlegend=False,
            hoverinfo="skip", row=1, col=column,
        )
        fig.add_bar(x=lags, y=estimates[1:], width=0.25, row=1, col=column)
        fig.add_hline(y=0, line_color="gray", line_width=1, row=1, col=column)

    fig.update_xaxes(title_text="Лаг, месяцев", dtick=6)
    fig.update_yaxes(title_text="Корреляция", range=[-1.05, 1.05])
    fig.update_layout(
        title=title,
        showlegend=False,
        height=420,
        margin=dict(t=90, b=70, l=70, r=50),
    )
    fig.show()

In [ ]:
sales_parts = pd.DataFrame({"Train": train, "Test": test})
show_series(sales_parts, "Продажи магазинов одежды в США", split=test.index[0])

## 1. STL

**Документация:** [statsmodels.STL](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.STL.html)

STL выделяет тренд с циклами $T_t$, сезонность $S_t$ и остаток $R_t$

$$
y_t = T_t + S_t + R_t
$$


**Другие библиотеки:**

- [sktime: STLTransformer](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.transformations.series.detrend.STLTransformer.html) — интерфейс к STL из statsmodels

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(train, period=period).fit()

components = pd.DataFrame(
    {
        "Ряд": train,
        "Тренд": stl.trend,
        "Сезонность": stl.seasonal,
        "Остаток": stl.resid,
    }
)

In [ ]:
def show_components(frame, title, units):
    show_series(frame, title, units, panels=True)

In [ ]:
show_components(components, "STL: продажи на train", "Млн USD")

## 2. Box–Cox

**Документация:** [SciPy: boxcox](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.boxcox.html), [inv_boxcox](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.inv_boxcox.html)

Преобразование может стабилизировать амплитуду колебаний, если она растет с уровнем ряда. Требуются положительные значения

$$
z_t=\begin{cases}(y_t^\lambda-1)/\lambda,&\lambda\ne0\\\log y_t,&\lambda=0\end{cases}
$$

Без `lmbda` функция `boxcox` оценивает параметр по train методом максимального правдоподобия

**Другие библиотеки:**

- [scikit-learn: PowerTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html) — `method="box-cox"`; дополнительную стандартизацию отключает `standardize=False`
- [Darts: BoxCox](https://unit8co.github.io/darts/generated_api/darts.dataprocessing.transformers.boxcox.html) — преобразование временного ряда

In [ ]:
from scipy.stats import boxcox
from scipy.special import inv_boxcox

# По train оцениваем lambda методом максимального правдоподобия и преобразуем значения
train_bc, lambda_hat = boxcox(train)
train_log = np.log(train).rename("Логарифм")  # частный случай Box–Cox при lambda=0

In [ ]:
lambda_hat

In [ ]:
transformation_comparison = pd.DataFrame(
    {
        "Исходный ряд": train,
        "log(y)": train_log,
        "Box–Cox": train_bc,
    }
)
show_components(
    transformation_comparison,
    "Один ряд в трех шкалах: сравнивайте форму, а не высоту",
    "Своя шкала",
)

In [ ]:
restored = inv_boxcox(train_bc, lambda_hat)  # Возвращаем исходные значения с той же lambda
abs(restored - train).max()  # Самое большое расхождение с исходными значениями

## 3. Экспоненциальное сглаживание и ETS

### SES: уровень

**Документация:** [statsmodels.SimpleExpSmoothing](https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.SimpleExpSmoothing.html)

После каждого наблюдения обновляем уровень $\ell_t$. Параметр $\alpha$ задает вес нового значения

$$
\ell_t=\alpha y_t+(1-\alpha)\ell_{t-1},\qquad
\widehat{y}_{t+h\mid t}=\ell_t
$$

Все прогнозы SES из одной точки одинаковы, $h$ — горизонт

Перед оценкой моделей переводим продажи из млн в млрд USD

**Другие библиотеки:**

- [StatsForecast: SimpleExponentialSmoothing, SimpleExponentialSmoothingOptimized](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html) — фиксированный или оцениваемый параметр сглаживания
- [Darts: ExponentialSmoothing](https://unit8co.github.io/darts/generated_api/darts.models.forecasting.exponential_smoothing.html) — без тренда и сезонности; интерфейс к statsmodels

In [ ]:
from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt,
    ExponentialSmoothing,
)

# Переводим млн USD в млрд USD; форма ряда и сезонность сохраняются
train_scaled = train / 1000

In [ ]:
ses = SimpleExpSmoothing(train_scaled).fit()  # Оцениваем вес нового наблюдения по train
print(f"SES: alpha = {ses.params['smoothing_level']}\n"
      f"Начальный уровень = {ses.params['initial_level']}")

### Хольт: уровень и тренд

**Документация:** [statsmodels.Holt](https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.Holt.html) — в том числе `damped_trend=True`

$b_t$ — наклон, $\beta$ — параметр его сглаживания

$$
\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})
$$

$$
b_t=\beta(\ell_t-\ell_{t-1})+(1-\beta)b_{t-1},\qquad
\widehat{y}_{t+h\mid t}=\ell_t+h b_t
$$

**Другие библиотеки:**

- [StatsForecast: Holt](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html) — метод Хольта
- [Darts: ExponentialSmoothing](https://unit8co.github.io/darts/generated_api/darts.models.forecasting.exponential_smoothing.html) — с трендом; интерфейс к statsmodels

In [ ]:
holt = Holt(train_scaled).fit()
print(
    f"Хольт: alpha = {holt.params['smoothing_level']}\n"
    f"beta = {holt.params['smoothing_trend']}\n"
    f"Начальный уровень = {holt.params['initial_level']}\n"
    f"Начальный тренд = {holt.params['initial_trend']}"
)

In [ ]:
# посмотреть как вернуть l_0

In [ ]:
def show_forecasts(frame, title):
    actual = pd.concat([train.iloc[-24:], test]).rename("Факт")
    forecast_comparison = pd.concat([actual, frame], axis=1)
    show_series(forecast_comparison, title, split=test.index[0])

In [ ]:
trend_predictions = pd.DataFrame(
    {
        "SES": ses.forecast(horizon) * 1000,
        "Хольт": holt.forecast(horizon) * 1000,
    }
)
show_forecasts(trend_predictions, f"Прогноз из конца train на {horizon} месяцев")

### Хольт–Уинтерс: добавляем сезонность

**Документация:** [statsmodels.ExponentialSmoothing](https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.ExponentialSmoothing.html)

- `trend="add"` — аддитивный тренд
- `seasonal="add"` — сезонные отклонения в единицах ряда
- `seasonal="mul"` — сезонные множители, нужны положительные значения
- `seasonal_periods=12` — сезон длиной в год

**Другие библиотеки:**

- [StatsForecast: HoltWinters](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html) — сезонное экспоненциальное сглаживание
- [Darts: ExponentialSmoothing](https://unit8co.github.io/darts/generated_api/darts.models.forecasting.exponential_smoothing.html) — с сезонностью; интерфейс к statsmodels

In [ ]:
hw_add = ExponentialSmoothing(
    train_scaled,
    trend="add",
    seasonal="add",
    seasonal_periods=period,
).fit()

print(
    f"Хольт–Уинтерс (add): alpha = {hw_add.params['smoothing_level']}, "
    f"beta = {hw_add.params['smoothing_trend']}, "
    f"gamma = {hw_add.params['smoothing_seasonal']}\n"
    f"Начальный уровень = {hw_add.params['initial_level']}\n"
    f"Начальный тренд = {hw_add.params['initial_trend']}\n"
    f"Начальные сезонные компоненты = {hw_add.params['initial_seasons']}"
)

# Меняем только тип сезонности
hw_mul = ExponentialSmoothing(
    train_scaled,
    trend="add",
    seasonal="mul",
    seasonal_periods=period,
).fit()

print(
    f"\n\nХольт–Уинтерс (mul): alpha = {hw_mul.params['smoothing_level']}, "
    f"beta = {hw_mul.params['smoothing_trend']}, "
    f"gamma = {hw_mul.params['smoothing_seasonal']}\n"
    f"Начальный уровень = {hw_mul.params['initial_level']}\n"
    f"Начальный тренд = {hw_mul.params['initial_trend']}\n"
    f"Начальные сезонные множители = {hw_mul.params['initial_seasons']}"
)


Как интерпретировать полученные значения параметров:

α ≈ 1 — уровень почти полностью обновляется по последнему наблюдению с учетом сезонности

β = 0 — наклон тренда не обновляется

γ ≈ 0 — сезонные множители почти не меняются. Сезонность остается; ее абсолютный размах меняется вместе с уровнем ряда

In [ ]:
predictions = trend_predictions.copy()
predictions["Хольт–Уинтерс: add"] = hw_add.forecast(horizon) * 1000
predictions["Хольт–Уинтерс: mul"] = hw_mul.forecast(horizon) * 1000

last_season = train.iloc[-period:].to_numpy()
predictions["Сезонный наивный"] = np.resize(last_season, horizon)

seasonal_predictions = predictions[
    ["Хольт–Уинтерс: add", "Хольт–Уинтерс: mul", "Сезонный наивный"]
]
show_forecasts(seasonal_predictions, "Сезонные прогнозы и знакомый бейзлайн")

### Обозначения ETS

**Документация:** [statsmodels.ETSModel](https://www.statsmodels.org/stable/generated/statsmodels.tsa.exponential_smoothing.ets.ETSModel.html)

Три позиции ETS — **ошибка, тренд, сезонность**. A — аддитивная компонента, M — мультипликативная, N — отсутствует, Ad — затухающий тренд

| Метод | Пример ETS |
|---|---|
| SES | ETS(A, N, N) |
| Хольт | ETS(A, A, N) |
| Хольт–Уинтерс: add | ETS(A, A, A) |
| Хольт–Уинтерс: mul | ETS(M, A, M) |

Пример вызова смотри в Приложении ★ 6. ETSModel и прогнозный интервал

**Другие библиотеки:**

- [StatsForecast: AutoETS](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html) — ETS с выбором структуры

### Сравним модели

In [ ]:
# Сравниваем каждый месяц train с тем же месяцем прошлого года
seasonal_changes = train.diff(period).dropna()
scale = seasonal_changes.abs().mean()  # Средняя ошибка сезонного наивного прогноза на train

scores = []
for name in predictions:
    error = test - predictions[name]  # Ошибка: факт минус прогноз на ту же дату
    absolute_error = error.abs()  # Ошибки разных знаков не погашают друг друга

    mae = absolute_error.mean()
    rmse = np.sqrt(np.mean(error ** 2))
    wape = 100 * absolute_error.sum() / test.abs().sum()
    mase = mae / scale  # Масштабируем MAE по сезонному ориентиру, рассчитанному на train

    scores.append(
        {"Метод": name, "MAE": mae, "RMSE": rmse, "WAPE, %": wape, "MASE": mase}
    )

scores = pd.DataFrame(scores)
scores = scores.set_index("Метод").sort_values("MAE")
display(scores)

## 4. Стационарность, разности, ADF / KPSS

При слабой стационарности среднее постоянно, дисперсия конечна, ковариация зависит только от лага $h$

$$
\mathbb E[y_t]=\mu,\qquad \operatorname{Var}(y_t)=\gamma(0)<\infty,
\qquad \operatorname{Cov}(y_t,y_{t-h})=\gamma(h)
$$

Сравним исходный ряд и его разности, затем проверим результаты ADF и KPSS

### Разности: метод `diff()`

Дифференцирование — переход от значений ряда к их разностям. Используем [`diff()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html): из текущего значения вычитаем предыдущее

Пример: `[10, 13, 12]` → `[NaN, 3, -1]`. Для первой точки нет предыдущего значения

`diff(12)` сравнивает месяц с тем же месяцем прошлого года. `dropna()` убирает начальные пропуски

In [ ]:
first_diff = train.diff().dropna()  # Изменение продаж относительно предыдущего месяца
season_diff = train.diff(12).dropna()  # Изменение относительно того же месяца прошлого года

In [ ]:
variants = {
    "Исходный ряд": train,
    "Обычная разность": first_diff,
    "Сезонная разность": season_diff,
}
variants_frame = pd.DataFrame(variants)
show_components(variants_frame, "Исходный ряд и его разности", "Млн USD")

### ADF и KPSS

**Документация:** [statsmodels.adfuller](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html), [statsmodels.kpss](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.kpss.html)

| Тест | Нулевая гипотеза H0 | Если p ≤ 0.05 |
|---|---|---|
| ADF | Есть единичный корень | Отвергаем единичный корень |
| KPSS | Стационарность вокруг уровня | Отвергаем стационарность вокруг уровня |

**Другие библиотеки:**

- [arch: ADF](https://arch.readthedocs.io/en/latest/unitroot/generated/arch.unitroot.ADF.html) — расширенный тест Дики–Фуллера
- [arch: KPSS](https://arch.readthedocs.io/en/latest/unitroot/generated/arch.unitroot.KPSS.html) — тест KPSS

In [ ]:
adf_first = adfuller(train)
adf_p = adf_first[1]  # Второй элемент результата — p-value
print("ADF: p-value =", adf_p)

if adf_p <= 0.05:
    print("Отвергаем гипотезу единичного корня: результат в пользу стационарности.")
else:
    print("Не отвергаем гипотезу единичного корня: ADF не подтвердил стационарность.")

In [ ]:
kpss_first = kpss(train)
kpss_p = kpss_first[1]
print("KPSS: p-value =", kpss_p)

if kpss_p <= 0.05:
    print("Отвергаем стационарность вокруг уровня.")
else:
    print("Не отвергаем стационарность вокруг уровня. Это не доказывает ее.")

In [ ]:
def stationarity_tests(values):
    values = values.dropna()
    adf_result = adfuller(values)

    kpss_result = kpss(values)

    adf_p = adf_result[1]
    kpss_p = kpss_result[1]

    # У тестов разные нулевые гипотезы, поэтому сравниваем оба результата
    if adf_p <= 0.05 and kpss_p > 0.05:
        conclusion = "Оба результата в пользу стационарности"
    elif adf_p > 0.05 and kpss_p <= 0.05:
        conclusion = "Оба результата в пользу нестационарности"
    elif adf_p <= 0.05 and kpss_p <= 0.05:
        conclusion = "Тесты расходятся: проверяем тренд и сдвиги уровня"
    else:
        conclusion = "Однозначного вывода нет: оба теста не отвергли H0"

    return {
        "ADF: p": adf_p,
        "KPSS: p": kpss_p,
        "Вывод при 5%": conclusion,
    }

In [ ]:
test_rows = {}
for name, values in variants.items():
    test_rows[name] = stationarity_tests(values)

test_results = pd.DataFrame(test_rows).T
display(test_results)

## 5. ACF / PACF

**Документация:** [statsmodels.acf](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.acf.html), [statsmodels.pacf](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.pacf.html)

**ACF** показывает линейную связь ряда с его прошлыми значениями. **PACF** исключает линейный вклад промежуточных лагов.

Сравним исходный ряд и сезонную разность. На месячных данных лаг 1 — предыдущий месяц, лаг 12 — тот же месяц прошлого года

Вызов `acf()` считает ACF, `pacf()` — PACF. `nlags=24` задает число лагов, `alpha=0.05` — 95% доверительные границы. Каждый вызов возвращает оценки и границы

Сначала выполняем расчет, затем передаем готовые результаты функции отрисовки

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

acf_result = acf(train, nlags=48, alpha=0.05)
pacf_result = pacf(train, nlags=48, alpha=0.05, method="ywm")

In [ ]:
show_acf_pacf(acf_result, pacf_result, "Продажи в исходной шкале")

In [ ]:
acf_result = acf(season_diff, nlags=48, alpha=0.05)
pacf_result = pacf(season_diff, nlags=48, alpha=0.05, method="ywm")

In [ ]:
show_acf_pacf(acf_result, pacf_result, "Сезонная разность продаж")

Голубая область — приближенные поточечные 95% доверительные границы. Для ACF используется формула Бартлетта, поэтому ширина может зависеть от лага; это не общая проверка всех лагов

Большая ACF уровней может отражать тренд и сезонность. Один столбец за границей не доказывает зависимость, внутри — ее отсутствие

<a id="appendix"></a>

## Приложение ★

1. Устойчивый STL
2. MSTL на синтетическом ряду
3. MSTL на реальных данных
4. SES: фиксированные значения alpha
5. Хольт с затуханием
6. ETSModel и прогнозный интервал

### ★ 1. Устойчивый STL

**Вопрос:** как провал 2020 года распределился между трендом, сезонностью и остатком обычного STL?

Теперь добавим `robust=True` и посмотрим полное разложение на всем train

In [ ]:
# Уменьшаем влияние необычных наблюдений на компоненты
stl_robust = STL(train, period=period, robust=True).fit()

In [ ]:
robust_components = pd.DataFrame(
    {
        "Ряд": train,
        "Тренд": stl_robust.trend,
        "Сезонность": stl_robust.seasonal,
        "Остаток": stl_robust.resid,
    }
)
show_components(robust_components, "Устойчивый STL: полное разложение на train", "Млн USD")

In [ ]:
comparison = pd.DataFrame(
    {
        "Обычный STL": stl.resid,
        "Устойчивый STL": stl_robust.resid,
    }
)
show_series(comparison, "Остатки обычного и устойчивого STL на всем train")

### ★ 2. MSTL на синтетическом ряду

**Документация:** [statsmodels.MSTL](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.MSTL.html)

Создадим синтетический почасовой ряд с суточной и недельной сезонностью

$$
x_t=100+0.01t+8\sin(2\pi t/24)+10\cos(2\pi t/168)+\varepsilon_t,
\qquad \varepsilon_t\sim N(0,1)
$$

$t$ — номер часа. Периоды — 24 и 168 часов, длина ряда — 5 недель

MSTL по очереди обновляет сезонности с помощью STL. Для первого вызова укажем только ряд и два периода

**Другие библиотеки:**

- [StatsForecast: MSTL](https://nixtlaverse.nixtla.io/statsforecast/src/core/models.html) — разложение и прогноз; для STL использует statsmodels

In [ ]:
from statsmodels.tsa.seasonal import MSTL

In [ ]:
hour = np.arange(24 * 7 * 5)
dates = pd.date_range("2025-01-01", periods=len(hour), freq="h")

hourly_trend = 100 + 0.01 * hour
daily_seasonality = 8 * np.sin(2 * np.pi * hour / 24)
weekly_seasonality = 10 * np.cos(2 * np.pi * hour / 168)
epsilon_t = np.random.normal(0, 1, len(hour))

x_t = hourly_trend + daily_seasonality + weekly_seasonality + epsilon_t
# Связываем значения с датами: MSTL сохранит временной индекс компонент
hourly = pd.Series(x_t, index=dates)

In [ ]:
mstl = MSTL(hourly, periods=(24, 168)).fit()

In [ ]:
mstl_components = pd.DataFrame(
    {
        "Ряд": hourly,
        "Тренд": mstl.trend,
        "Сутки, 24 часа": mstl.seasonal["seasonal_24"],
        "Неделя, 168 часов": mstl.seasonal["seasonal_168"],
        "Остаток": mstl.resid,
    }
)

show_components(mstl_components, "MSTL", "Усл ед")

### ★ 3. MSTL на реальных данных: спрос на электричество

Спрос на электричество в штате Виктория, Австралия, без части крупных промышленных потребителей. Источник — AEMO; берем январь–февраль 2012 года

В исходном CSV значения за полчаса, в МВт·ч. Складываем два получасовых значения и получаем потребление за час. Для почасового ряда суточный и недельный периоды — 24 и 168

[Описание данных](https://tsibbledata.tidyverts.org/reference/vic_elec.html) · [Пример MSTL в statsmodels](https://www.statsmodels.org/stable/examples/notebooks/generated/mstl_decomposition.html)

**Документация:** [statsmodels.MSTL](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.MSTL.html)

MSTL по очереди выделяет сезонные компоненты с помощью STL. Буква M означает несколько сезонностей, а не мультипликативность

In [ ]:
from statsmodels.tsa.seasonal import MSTL

In [ ]:
electricity_url = (
    "https://raw.githubusercontent.com/tidyverts/tsibbledata/"
    "master/data-raw/vic_elec/VIC2015/demand.csv"
)
electricity_raw = pd.read_csv(electricity_url)

calendar_date = pd.to_datetime(
    electricity_raw["Date"], unit="D", origin="1899-12-30"
)
minutes_from_midnight = (electricity_raw["Period"] - 1) * 30
time_within_day = pd.to_timedelta(minutes_from_midnight, unit="min")
electricity_raw["Дата"] = calendar_date + time_within_day

electricity_raw = electricity_raw.set_index("Дата").sort_index()
half_hour_demand = electricity_raw["OperationalLessIndustrial"]
half_hour_demand = half_hour_demand.loc["2012-01":"2012-02"]

electricity = half_hour_demand.resample("h").sum(min_count=2)
electricity.name = "Спрос за час"

In [ ]:
show_series(
    electricity.to_frame(),
    "Виктория: почасовой спрос на электричество, январь–февраль 2012",
    units="МВт·ч",
)

In [ ]:
electricity_mstl = MSTL(electricity, periods=(24, 168)).fit()

electricity_components = pd.DataFrame(
    {
        "Ряд": electricity,
        "Тренд": electricity_mstl.trend,
        "Сутки, 24 часа": electricity_mstl.seasonal["seasonal_24"],
        "Неделя, 168 часов": electricity_mstl.seasonal["seasonal_168"],
        "Остаток": electricity_mstl.resid,
    }
)
show_components(electricity_components, "MSTL: реальный спрос на электричество", "МВт·ч")

In [ ]:
from statsmodels.tsa.stattools import acf

electricity_max_lag = 24 * 14  # Смотрим зависимость в пределах двух недель
electricity_acf = acf(electricity_mstl.resid, nlags=electricity_max_lag)
electricity_lags = np.arange(1, electricity_max_lag + 1)

In [ ]:
fig = go.Figure()
fig.add_bar(
    x=electricity_lags,
    y=electricity_acf[1:],
)
fig.add_hline(y=0, line_color="gray", line_width=1)
fig.update_layout(
    title="ACF остатков MSTL: спрос на электричество",
    xaxis_title="Лаг, часов",
    yaxis_title="ACF",
    xaxis=dict(dtick=24),
    yaxis=dict(range=[-1.05, 1.05]),
    height=420,
    showlegend=False,
    margin=dict(t=80, b=70, l=70, r=45),
)
fig.show()

### ★ 4. SES: как влияет alpha

Теперь зададим два значения alpha вручную, оставив одинаковый начальный уровень

In [ ]:
ses_fixed = {}
initial_level = train_scaled.iloc[0]  # Общее начальное значение для сравнения

for alpha in [0.2, 0.8]:
    model = SimpleExpSmoothing(
        train_scaled,
        initialization_method="known",
        initial_level=initial_level,
    )
    fit = model.fit(smoothing_level=alpha, optimized=False)
    ses_fixed[alpha] = fit
    print(f"SES с заданной alpha: {fit.params['smoothing_level']}")

In [ ]:
ses_levels = pd.DataFrame({"Факт": train.iloc[-36:]})

for alpha, fit in ses_fixed.items():
    level = fit.level * 1000  # Уровень уже имеет даты; возвращаем млн USD
    ses_levels[f"Уровень, alpha={alpha}"] = level

show_series(ses_levels, "SES: уровень после получения текущего наблюдения")

### ★ 5. Хольт с затуханием

**Документация:** [statsmodels.Holt](https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.Holt.html)

Добавим `damped_trend=True`. При $0<\phi<1$ вклад наклона в прогноз равен $(\phi+\ldots+\phi^h)b_t$: с горизонтом рост замедляется

In [ ]:
holt_damped = Holt(train_scaled, damped_trend=True).fit()
print(
    f"Хольт с затуханием: alpha = {holt_damped.params['smoothing_level']}, "
    f"beta = {holt_damped.params['smoothing_trend']}, "
    f"phi = {holt_damped.params['damping_trend']}"
)

In [ ]:
damped_predictions = pd.DataFrame(
    {
        "SES": ses.forecast(horizon) * 1000,
        "Хольт": holt.forecast(horizon) * 1000,
        "Хольт с затуханием": holt_damped.forecast(horizon) * 1000,
    }
)
show_forecasts(damped_predictions, f"Прогноз из конца train на {horizon} месяцев")

In [ ]:
predictions_with_damping = predictions.copy()
predictions_with_damping["Хольт с затуханием"] = holt_damped.forecast(horizon) * 1000

In [ ]:
# Сравниваем каждый месяц train с тем же месяцем прошлого года
seasonal_changes = train.diff(period).dropna()
scale = seasonal_changes.abs().mean()  # Средняя ошибка сезонного наивного прогноза на train

scores_with_damping = []
for name in predictions_with_damping:
    error = test - predictions_with_damping[name]  # Ошибка: факт минус прогноз на ту же дату
    absolute_error = error.abs()  # Ошибки разных знаков не погашают друг друга

    mae = absolute_error.mean()
    rmse = np.sqrt(np.mean(error ** 2))
    wape = 100 * absolute_error.sum() / test.abs().sum()
    mase = mae / scale  # Масштабируем MAE по сезонному ориентиру, рассчитанному на train

    scores_with_damping.append(
        {"Метод": name, "MAE": mae, "RMSE": rmse, "WAPE, %": wape, "MASE": mase}
    )

scores_with_damping = pd.DataFrame(scores_with_damping)
scores_with_damping = scores_with_damping.set_index("Метод").sort_values("MAE")
display(scores_with_damping)

### ★ 6. ETSModel и прогнозный интервал

**Документация:** [statsmodels.ETSModel](https://www.statsmodels.org/stable/generated/statsmodels.tsa.exponential_smoothing.ets.ETSModel.html)

Построим 95% прогнозный интервал для ETS(A, N, N). В модели нет сезонности, и интервал ее не заменяет

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

ets_ses = ETSModel(train_scaled).fit(disp=False)  # По умолчанию ETS(A,N,N)
# Связываем названия оцененных параметров с их значениями
ets_parameters = dict(zip(ets_ses.model.param_names, ets_ses.params))
print(f"ETS(A,N,N): alpha = {ets_parameters['smoothing_level']}")

forecast_start = len(train)
forecast_end = forecast_start + horizon - 1  # Правая граница включается
ets_prediction = ets_ses.get_prediction(start=forecast_start, end=forecast_end)
ets_summary = ets_prediction.summary_frame() * 1000  # Возвращаем млн USD

In [ ]:
sigma = np.sqrt(ets_ses.scale)
print(f"ETS(A,N,N): sigma = {sigma} млрд USD")

In [ ]:
fig = go.Figure()
fig.add_scatter(
    x=test.index,
    y=ets_summary["pi_upper"],
    mode="lines",
    line_width=0,
    showlegend=False,
)
fig.add_scatter(
    x=test.index,
    y=ets_summary["pi_lower"],
    mode="lines",
    line_width=0,
    fill="tonexty",
    fillcolor="rgba(31,119,180,0.18)",
    name="95% прогнозный интервал",
)
fig.add_scatter(x=test.index, y=ets_summary["mean"], name="ETS(A,N,N)")
fig.add_scatter(x=test.index, y=test, name="Факт", line_color="gray", opacity=0.6)
fig.update_layout(
    title="ETS: точечный прогноз и прогнозный интервал",
    xaxis_title="Дата",
    yaxis_title="Продажи, млн USD",
    height=420,
)
fig.show()